# Stack Overflow Developer Survey 2012 - canonical (not yet encoded) version

**Purpose:** same treatment as the 2011 notebook, applied to the 2012 file. Produces `canonical_2012.csv`: shared column names, single-answer columns kept exactly as the respondent chose them, multi-answer questions stored as `;`-separated lists, the language write-ins cleaned into canonical names with a count. **No one-hot encoding, no imputation, no scaling, no rare-level merging.**

**Rows kept:** only respondents with a usable salary band (blank, `Rather not say` and `Student / Unemployed` are dropped, see `CONFIG`).

**What differs from 2011** (all checked in the cells below): 75 columns instead of 64; `Objective-C` and `HTML5` are now tick-boxes in the language question; the budget question is a single column (no conflicting ticks to resolve); Stack Overflow Careers, advertising, reputation and extra-site columns exist; there is no recommendation column; the experience column has different Excel date serials (`40944`, `41070`).

## Gate 1 - Load and normalize
### Cell 1.0 - Imports, file path, decision log

In [20]:
import csv, re, math, datetime
from pathlib import Path
import numpy as np
import pandas as pd

pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 70)
pd.set_option("display.max_rows", 200)

CANDIDATES = [Path("data/2012/results.csv"), Path("2012_results.csv"), Path("results.csv")]
PATH = next((p for p in CANDIDATES if p.exists()), None)
if PATH is None:
    raise FileNotFoundError("none of the candidate paths exist: " + ", ".join(str(p) for p in CANDIDATES))
print("using file:", PATH, "| size in bytes:", PATH.stat().st_size)
print("pandas version:", pd.__version__)

decision_log = []
def log(step, text):
    decision_log.append((step, text))
    print("[LOG]", step, "-", text)

using file: data\2012\results.csv | size in bytes: 2521440
pandas version: 3.1.0


### Cell 1.1 - Detect the file encoding
UTF-8 is tried first; the 2012 file fails on a byte (0x8e), so `cp1252` is used, and the byte that forced the fallback is printed.

In [21]:
raw_bytes = PATH.read_bytes()
non_ascii = sum(1 for b in raw_bytes if b > 127)
print("total bytes:", len(raw_bytes), "| non-ASCII bytes:", non_ascii)
try:
    raw_bytes.decode("utf-8")
    ENCODING = "utf-8"
    print("utf-8 decodes the whole file cleanly")
except UnicodeDecodeError as err:
    ENCODING = "cp1252"
    print("utf-8 fails at byte", err.start, "- surrounding bytes:", raw_bytes[max(0, err.start - 25):err.start + 10])
    print("falling back to cp1252")
log("1.1 encoding", "file read with encoding=" + ENCODING + " (" + str(non_ascii) + " non-ASCII byte(s) in the file)")

total bytes: 2521440 | non-ASCII bytes: 8
utf-8 fails at byte 127259 - surrounding bytes: b'intention to put my resum\x8e in the p'
falling back to cp1252
[LOG] 1.1 encoding - file read with encoding=cp1252 (8 non-ASCII byte(s) in the file)


### Cell 1.2 - Load as text, without pandas' automatic NA parsing
`dtype=str, keep_default_na=False, na_values=[]` so that real answers such as `None` survive. Blanks are converted in Cell 1.5.

In [22]:
df = pd.read_csv(PATH, header=0, encoding=ENCODING, dtype=str, keep_default_na=False, na_values=[])
print("shape after load (rows, columns):", df.shape)
if all(str(c).isdigit() for c in df.columns):
    raise ValueError("header row was consumed as data - reload")
print("first 3 column names:", list(df.columns[:3]))
log("1.2 load", "loaded " + str(df.shape[0]) + " rows x " + str(df.shape[1]) + " columns, all as text, default NA parsing off")

shape after load (rows, columns): (6244, 75)
first 3 column names: ['What Country or Region do you live in?', 'Which US State or Territory do you live in?', 'How old are you?']
[LOG] 1.2 load - loaded 6244 rows x 75 columns, all as text, default NA parsing off


### Cell 1.3 - Cross-check with the `csv` module
Confirms pandas did not drop or merge rows, and lists the truly empty header cells (continuations of the question to their left).

In [23]:
with open(PATH, encoding=ENCODING, newline="") as fh:
    csv_rows = list(csv.reader(fh))
print("rows seen by csv module (header + data):", len(csv_rows))
print("distinct row lengths:", sorted(set(len(r) for r in csv_rows)))
assert len(csv_rows) - 1 == len(df), "pandas and csv module disagree on row count"
assert all(len(r) == df.shape[1] for r in csv_rows), "ragged rows found"
header_raw = csv_rows[0]
blank_header_positions = [i for i, h in enumerate(header_raw) if h.strip() == ""]
print("header cells that are empty:", len(blank_header_positions), "of", len(header_raw))

rows seen by csv module (header + data): 6245
distinct row lengths: [75]
header cells that are empty: 51 of 75


### Cell 1.4 - Strip whitespace from column names

In [24]:
changed = [(i, repr(c)) for i, c in enumerate(df.columns) if c != c.strip()]
print("column names with leading/trailing whitespace:", changed)
df.columns = df.columns.astype(str).str.strip()
print("duplicate column names:", df.columns[df.columns.duplicated()].tolist())
log("1.4 strip", str(len(changed)) + " column name(s) had surrounding whitespace and were stripped")

column names with leading/trailing whitespace: [(38, "'What best describes your career / job satisfaction? '"), (64, "'In the last 12 months, how much money have you spent on personal technology-related purchases? '")]
duplicate column names: []
[LOG] 1.4 strip - 2 column name(s) had surrounding whitespace and were stripped


### Cell 1.5 - Blank cells to NaN
Empty and whitespace-only cells become NaN, once. Literal text such as `None` is **not** touched; it is listed so you can see it.

In [25]:
empty_exact = int((df == "").sum().sum())
ws_only = int(df.apply(lambda s: s.str.match(r"^\s+$")).sum().sum())
na_before = int(df.isna().sum().sum())
df = df.replace(r"^\s*$", np.nan, regex=True)
na_after = int(df.isna().sum().sum())
print("cells exactly empty:", empty_exact, "| whitespace-only:", ws_only, "| NaN increase:", na_after - na_before)
assert na_after - na_before == empty_exact + ws_only
print("share missing: %.1f%%" % (100 * na_after / df.size))
literals = ["None", "N/A", "NA", "n/a", "null", "NULL", "NaN", "nan"]
found = [(i, col[:50], int(df.iloc[:, i].isin(literals).sum())) for i, col in enumerate(df.columns)]
found = [f for f in found if f[2]]
print("literal NA-like text kept as real values:")
for item in found:
    print("   col", item[0], "|", item[1], "|", item[2], "cells")
log("1.5 blanks", str(empty_exact + ws_only) + " empty/whitespace cells converted to NaN; " + str(sum(f[2] for f in found)) + " literal 'None'-style cells kept as text")

cells exactly empty: 283328 | whitespace-only: 0 | NaN increase: 283328
share missing: 60.5%
literal NA-like text kept as real values:
   col 35 | Unnamed: 35 | 6 cells
   col 43 | Unnamed: 43 | 1 cells
   col 63 | Unnamed: 63 | 1 cells
   col 71 | What advertisers do you remember seeing on Stack O | 100 cells
   col 73 | Which of our sites do you frequent most? | 52 cells
[LOG] 1.5 blanks - 283328 empty/whitespace cells converted to NaN; 160 literal 'None'-style cells kept as text


### Cell 1.6 - Separate the option-label row
Row 0 of the data holds the option label of each column (`Influencer`, `Java`, `iPhone`, ...) or `Response`. It is not a respondent, so it moves to `option_labels`. The original index is kept: index `n` is data row `n` of the file.

In [26]:
first_row = df.iloc[0]
print("non-empty cells in row 0:", int(first_row.notna().sum()), "of", df.shape[1], "| 'Response' cells:", int((first_row == "Response").sum()))
for i, v in enumerate(first_row):
    if v != "Response":
        print("   col", i, "->", v)
rows_before = len(df)
option_labels = df.iloc[0].copy()
df = df.iloc[1:].copy()
assert len(df) == rows_before - 1
log("1.6 option row", "row 0 moved to option_labels (rows " + str(rows_before) + " -> " + str(len(df)) + ")")

non-empty cells in row 0: 75 of 75 | 'Response' cells: 18
   col 7 -> Influencer
   col 8 -> Recommender
   col 9 -> Approver
   col 10 -> Purchaser
   col 11 -> Check Writer
   col 12 -> No Involvement
   col 13 -> I'm a Seller
   col 14 -> Hardware
   col 15 -> Servers
   col 16 -> Software
   col 17 -> User Equipment: Monitors, PCs, Laptops
   col 18 -> Consultants
   col 19 -> Other
   col 22 -> Java
   col 23 -> JavaScript
   col 24 -> CSS
   col 25 -> PHP
   col 26 -> Python
   col 27 -> Objective-C
   col 28 -> Ruby
   col 29 -> SQL
   col 30 -> C#
   col 31 -> C++
   col 32 -> C
   col 33 -> Perl
   col 34 -> HTML5
   col 35 -> None
   col 36 -> Other (please specify)
   col 43 -> Other (please specify)
   col 44 -> iPhone
   col 45 -> Android
   col 46 -> Blackberry
   col 47 -> Windows Phone
   col 48 -> Other Smart Phone
   col 49 -> Regular Mobile Phone
   col 50 -> Kindle
   col 51 -> Nook
   col 52 -> AppleTV
   col 53 -> Boxee
   col 54 -> Other media streaming device
  

### Cell 1.7 - Indexed column list (audit)
Every column with its index, header, row-0 label and fill rate. `col_index` is printed so the indices used in Gate 2 can be checked by eye.

In [27]:
records = []
parent_idx = None
for i, col in enumerate(df.columns):
    is_blank = i in blank_header_positions
    if not is_blank:
        parent_idx = i
    n_filled = int(df.iloc[:, i].notna().sum())
    records.append({"idx": i, "header": col[:62] if not is_blank else "(blank - continues col " + str(parent_idx) + ")",
                    "row0_label": option_labels.iloc[i], "n_filled": n_filled, "pct_missing": round(100 * (1 - n_filled / len(df)), 1)})
col_index = pd.DataFrame(records)
print(col_index.to_string(index=False))
filled_per_row = df.notna().sum(axis=1)
print("fully empty rows:", int((filled_per_row == 0).sum()))

 idx                                                         header                                      row0_label  n_filled  pct_missing
   0                         What Country or Region do you live in?                                        Response      6243          0.0
   1                    Which US State or Territory do you live in?                                        Response      1775         71.6
   2                                               How old are you?                                        Response      5964          4.5
   3       How many years of IT/Programming experience do you have?                                        Response      5964          4.5
   4 How would you best describe the industry you currently work in                                        Response      5964          4.5
   5                 Which best describes the size of your company?                                        Response      5737          8.1
   6         Which of the f

## Gate 2 - Freeze names, choices, junk rows
### Cell 2.0 - Column positions and `CONFIG`
Each position is asserted against a keyword from its question text. `CONFIG` holds the row-level choices.

**Decisions (review me):**
- `drop_student_unemployed`: the `Student / Unemployed` band is a status, not an amount (same as 2011).
- `drop_rather_not_say`: `Rather not say` is new in 2012's answers and is not a salary input, so those rows are dropped too. Set to `False` to keep them (their `ConvertedCompYearly` would be empty).
- `top_band_midpoint`: 150,000 for `>$140,000`, the same as 2011.

In [28]:
RAW = list(df.columns)
IDX = {"country": 0, "state": 1, "age": 2, "exp": 3, "industry": 4, "size": 5, "occ": 6, "budget": 20, "project": 21,
       "os": 37, "sat": 38, "salary": 39, "careers_aware": 40, "careers_profile": 41, "careers_why": 42, "careers_why_other": 43,
       "spend": 64, "ad_recall": 71, "rep": 72, "site": 73, "site_other": 74}
COL = {k: RAW[v] for k, v in IDX.items()}
KEYWORD = {"country": "country", "state": "us state", "age": "how old", "exp": "years of it", "industry": "industry",
           "size": "size of your company", "occ": "occupation", "budget": "budget", "project": "project", "os": "operating system",
           "sat": "satisfaction", "salary": "compensation", "careers_aware": "aware of stack overflow careers",
           "careers_profile": "careers profile", "careers_why": "elaborate why", "careers_why_other": "unnamed",
           "spend": "spent", "ad_recall": "advertisers", "rep": "reputation", "site": "sites", "site_other": "unnamed"}
for k, col in COL.items():
    assert KEYWORD[k] in col.lower(), "column " + repr(col) + " does not contain " + repr(KEYWORD[k])
    print("ok  idx", IDX[k], "|", k, "|", col[:70])

BLOCKS = {"purchase_involvement": (7, 14), "purchase_types": (14, 20), "language": (22, 37), "products": (44, 64), "ads": (65, 71)}
for key, (a, b) in BLOCKS.items():
    print("block", key, "| cols", a, "to", b - 1, "|", RAW[a][:70], "| labels:", list(option_labels.iloc[a:b])[:6], "...")

CONFIG = {"drop_student_unemployed": True, "drop_rather_not_say": True, "top_band_midpoint": 150000}
print(CONFIG)

row_log = [("1.6 option-label row moved out of data", len(csv_rows) - 1, len(df))]
def log_rows(step, before, after):
    row_log.append((step, before, after))
    print("[ROWS]", step, ":", before, "->", after, "(removed", before - after, ")")

ok  idx 0 | country | What Country or Region do you live in?
ok  idx 1 | state | Which US State or Territory do you live in?
ok  idx 2 | age | How old are you?
ok  idx 3 | exp | How many years of IT/Programming experience do you have?
ok  idx 4 | industry | How would you best describe the industry you currently work in?
ok  idx 5 | size | Which best describes the size of your company?
ok  idx 6 | occ | Which of the following best describes your occupation?
ok  idx 20 | budget | What is your budget for outside expenditures (hardware, software, cons
ok  idx 21 | project | What type of project are you developing?
ok  idx 37 | os | Which desktop operating system do you use the most?
ok  idx 38 | sat | What best describes your career / job satisfaction?
ok  idx 39 | salary | Including bonus, what is your annual compensation in USD?
ok  idx 40 | careers_aware | Have you visited / Are you aware of Stack Overflow Careers?
ok  idx 41 | careers_profile | Do you have a Stack Overflow Careers Prof

### Cell 2.1 - Junk rows: propose and apply
Respondents who answered nothing except Country / US State (US State is routing, not an answer). The cell prints how many of them have a salary band before dropping.

In [29]:
other_cols = [c for c in df.columns if c not in (COL["country"], COL["state"])]
is_junk = df[other_cols].notna().sum(axis=1) == 0
print("rows the rule would remove:", int(is_junk.sum()), "| of those with a salary band:", int((is_junk & df[COL["salary"]].notna()).sum()))
rows_before = len(df)
df = df.loc[~is_junk].copy()
log_rows("2.1 junk rows (only Country/State answered)", rows_before, len(df))
log("2.1 junk rows", str(rows_before - len(df)) + " rows removed")

rows the rule would remove: 279 | of those with a salary band: 0
[ROWS] 2.1 junk rows (only Country/State answered) : 6243 -> 5964 (removed 279 )
[LOG] 2.1 junk rows - 279 rows removed


## Gate 3 - Salary
### Cell 3.1 - Band table
`CompTotal` keeps the original band text; `ConvertedCompYearly` is the band midpoint in annual USD. The 2012 question says "Including bonus, what is your annual compensation in USD?", so it matches your "annual including bonus" requirement. The bands are ranges, so the midpoint is a first estimate (you said you will calculate the number later; the lower and upper bounds are printed so you can use another rule). Pay before 2019 is base salary only per your notes, so 2011-2015 should be treated as one convention.

In [30]:
STUDENT_LABEL = "Student / Unemployed"
RATHER_NOT = "Rather not say"
BANDS = [
    ("<$20,000", 0, 20000, 10000), ("$20,000 - $40,000", 20000, 40000, 30000), ("$40,000 - $60,000", 40000, 60000, 50000),
    ("$60,000 - $80,000", 60000, 80000, 70000), ("$80,000 - $100,000", 80000, 100000, 90000),
    ("$100,000 - $120,000", 100000, 120000, 110000), ("$120,000 - $140,000", 120000, 140000, 130000),
    (">$140,000", 140000, None, CONFIG["top_band_midpoint"]),
]
observed = set(df[COL["salary"]].dropna().unique())
known = set(b[0] for b in BANDS) | {STUDENT_LABEL, RATHER_NOT}
assert observed <= known, sorted(observed - known)
counts = df[COL["salary"]].value_counts()
print(pd.DataFrame([[b[0], int(counts.get(b[0], 0)), b[1], b[2], b[3]] for b in BANDS] +
                   [[STUDENT_LABEL, int(counts.get(STUDENT_LABEL, 0)), None, None, None], [RATHER_NOT, int(counts.get(RATHER_NOT, 0)), None, None, None]],
                   columns=["band", "n_rows", "lower", "upper", "midpoint_used"]).to_string(index=False))
print("missing salary band:", int(df[COL["salary"]].isna().sum()))

                band  n_rows    lower    upper  midpoint_used
            <$20,000     688      0.0  20000.0        10000.0
   $20,000 - $40,000     647  20000.0  40000.0        30000.0
   $40,000 - $60,000     691  40000.0  60000.0        50000.0
   $60,000 - $80,000     609  60000.0  80000.0        70000.0
  $80,000 - $100,000     479  80000.0 100000.0        90000.0
 $100,000 - $120,000     298 100000.0 120000.0       110000.0
 $120,000 - $140,000     147 120000.0 140000.0       130000.0
           >$140,000     181 140000.0      NaN       150000.0
Student / Unemployed     515      NaN      NaN            NaN
      Rather not say     982      NaN      NaN            NaN
missing salary band: 727


### Cell 3.2 - Drop rows with no usable salary
Blank, `Student / Unemployed` and `Rather not say` are removed (per `CONFIG`). Counts are logged for each step.

In [31]:
rows_before = len(df)
df = df.loc[df[COL["salary"]].notna()].copy()
log_rows("3.2 salary band blank", rows_before, len(df))
if CONFIG["drop_rather_not_say"]:
    rows_before = len(df)
    df = df.loc[df[COL["salary"]] != RATHER_NOT].copy()
    log_rows("3.2 'Rather not say' band", rows_before, len(df))
if CONFIG["drop_student_unemployed"]:
    rows_before = len(df)
    df = df.loc[df[COL["salary"]] != STUDENT_LABEL].copy()
    log_rows("3.2 Student / Unemployed band", rows_before, len(df))
assert df[COL["salary"]].isin([b[0] for b in BANDS]).all() or not (CONFIG["drop_rather_not_say"] and CONFIG["drop_student_unemployed"])
log("3.2 salary rows", "rows without a usable salary band removed; " + str(len(df)) + " remain")

[ROWS] 3.2 salary band blank : 5964 -> 5237 (removed 727 )
[ROWS] 3.2 'Rather not say' band : 5237 -> 4255 (removed 982 )
[ROWS] 3.2 Student / Unemployed band : 4255 -> 3740 (removed 515 )
[LOG] 3.2 salary rows - rows without a usable salary band removed; 3740 remain


## Gate 4 - Multi-answer questions stored as lists
### Cell 4.0 - Helper: option columns to a `;`-separated list
One column per option in 2012, written as one `;` list (the 2016+ format). Labels come from row 0; the free-text `Other (please specify)` label becomes `Other`; a respondent who ticked nothing gets NaN, not an empty string.

In [32]:
def block_to_list(frame, key):
    a, b = BLOCKS[key]
    cols = RAW[a:b]
    labels = [option_labels.iloc[i] for i in range(a, b)]
    nice = ["Other" if l.lower().startswith("other (please specify)") else l for l in labels]
    assert all(";" not in l for l in nice), "an option label contains ';'"
    for col, lab in zip(cols, labels):
        if not lab.lower().startswith("other (please specify)"):
            assert (frame[col].dropna() == lab).all(), "cell text differs from the option label in " + col
    per_row = frame[cols].notna().sum(axis=1)
    print("block:", key, "| cols", a, "to", b - 1, "| options:", nice)
    print("rows by number of options ticked:", per_row.value_counts().sort_index().to_dict())
    def join_row(values):
        picked = [n for n, v in zip(nice, values) if pd.notna(v)]
        return ";".join(picked) if picked else np.nan
    return frame[cols].apply(lambda r: join_row(r.values), axis=1)

### Cell 4.1 - Purchasing involvement, purchase types, products

In [33]:
purchase_involvement = block_to_list(df, "purchase_involvement")
print()
purchase_types = block_to_list(df, "purchase_types")
print()
tech_products = block_to_list(df, "products")
for name, s in [("PurchaseInvolvement", purchase_involvement), ("PurchaseTypes", purchase_types), ("TechProductsOwned", tech_products)]:
    print(name, "| blank:", int(s.isna().sum()), "| example:", s.dropna().iloc[0])
log("4.1 list blocks", "purchasing involvement, purchase types and products stored as ';' lists")

block: purchase_involvement | cols 7 to 13 | options: ['Influencer', 'Recommender', 'Approver', 'Purchaser', 'Check Writer', 'No Involvement', "I'm a Seller"]
rows by number of options ticked: {1: 2018, 2: 1038, 3: 393, 4: 160, 5: 86, 6: 32, 7: 13}

block: purchase_types | cols 14 to 19 | options: ['Hardware', 'Servers', 'Software', 'User Equipment: Monitors, PCs, Laptops', 'Consultants', 'Other']
rows by number of options ticked: {0: 1022, 1: 865, 2: 679, 3: 463, 4: 382, 5: 202, 6: 127}

block: products | cols 44 to 63 | options: ['iPhone', 'Android', 'Blackberry', 'Windows Phone', 'Other Smart Phone', 'Regular Mobile Phone', 'Kindle', 'Nook', 'AppleTV', 'Boxee', 'Other media streaming device', 'Netbook', 'PS3', 'Xbox', 'Wii', 'Other gaming system', 'Kindle Fire', 'iPad', 'Other tablet', 'Other']
rows by number of options ticked: {0: 160, 1: 720, 2: 857, 3: 706, 4: 548, 5: 331, 6: 202, 7: 110, 8: 56, 9: 20, 10: 12, 11: 10, 12: 2, 13: 3, 15: 1, 20: 2}
PurchaseInvolvement | blank: 0 | e

### Cell 4.2 - Budget for outside expenditures (range selection)
**Your rules:** if several ranges are ticked, keep the highest; if the person also said `Don't know`, it is `Don't know`.

**What 2012 looks like:** the budget question is one column here, not one column per range, so no respondent can have two ranges and no conflict exists. The rules are still coded below and counted (they fire 0 times), so the notebook is correct if a conflicting row ever appears. The column is kept as text, and blanks stay blank.

In [34]:
BUDGET_ORDER = ["<$10,000", "$10,001 - $25,000", "$25,001 - $40,000", "$41,000 - $75,000", "$75,001 - $100,000", "$100,001 - $150,000", ">$150,000"]
s = df[COL["budget"]]
unknown = sorted(set(s.dropna()) - set(BUDGET_ORDER) - {"Don't know"})
print("unexpected budget values:", unknown)
assert not unknown
print("cells containing more than one answer (separator found):", int(s.dropna().str.contains(";|\\|").sum()))
def resolve_budget(cell):
    # if a cell ever held several answers: Don't know wins, then the highest range
    parts = [p.strip() for p in re.split(r"[;|]", cell)] if isinstance(cell, str) else []
    if not parts:
        return np.nan
    if "Don't know" in parts:
        return "Don't know"
    return max(parts, key=BUDGET_ORDER.index)
budget = s.map(resolve_budget)
assert budget.equals(s) or (budget.fillna("x") == s.fillna("x")).all()
print(budget.value_counts(dropna=False).to_string())
log("4.2 budget", "single-answer column in 2012; 0 conflicting rows; Don't know / highest-range rules coded but not needed")

unexpected budget values: []
cells containing more than one answer (separator found): 0
What is your budget for outside expenditures (hardware, software, consulting, etc) for 2011?
Don't know             1100
NaN                    1022
<$10,000                834
$10,001 - $25,000       308
>$150,000               204
$25,001 - $40,000       108
$41,000 - $75,000        63
$100,001 - $150,000      53
$75,001 - $100,000       48
[LOG] 4.2 budget - single-answer column in 2012; 0 conflicting rows; Don't know / highest-range rules coded but not needed


## Gate 5 - Language proficiency (cols 22-36)
### Cell 5.1 - Look at the block
13 tick-boxes (`Java` ... `HTML5`, `None`) plus one free-text box. Note: `CSS` and `HTML5` are separate tick-boxes in 2012, and `Objective-C` is now a tick-box.

In [35]:
a, b = BLOCKS["language"]
lang_cols = RAW[a:b]
lang_labels = [option_labels.iloc[i] for i in range(a, b)]
for i, lab in zip(range(a, b), lang_labels):
    print("col", i, "->", lab, "| ticked:", int(df[RAW[i]].notna().sum()))
WRITEIN_COL = lang_cols[-1]
assert lang_labels[-1].lower().startswith("other (please specify)")
writein = df[WRITEIN_COL].dropna()
print()
print("write-ins:", len(writein), "| distinct as typed:", writein.nunique(), "| distinct lowercased:", writein.str.lower().str.strip().nunique())
print(writein.str.lower().str.strip().value_counts().head(15).to_string())

col 22 -> Java | ticked: 1525
col 23 -> JavaScript | ticked: 2194
col 24 -> CSS | ticked: 1972
col 25 -> PHP | ticked: 1278
col 26 -> Python | ticked: 793
col 27 -> Objective-C | ticked: 391
col 28 -> Ruby | ticked: 396
col 29 -> SQL | ticked: 2369
col 30 -> C# | ticked: 1759
col 31 -> C++ | ticked: 963
col 32 -> C | ticked: 945
col 33 -> Perl | ticked: 343
col 34 -> HTML5 | ticked: 1472
col 35 -> None | ticked: 2
col 36 -> Other (please specify) | ticked: 589

write-ins: 589 | distinct as typed: 297 | distinct lowercased: 257
Unnamed: 36
vb.net            49
delphi            42
groovy            23
actionscript      20
coldfusion        19
vb                17
scala             17
haskell           15
r                 11
matlab            10
vba                9
visual basic       9
actionscript 3     8
pl/sql             8
clojure            7


### Cell 5.2 - What the tick-boxes become (canonical names)
**Decisions (review me):** `CSS` and `HTML5` both become **`HTML/CSS`** (a person who ticked both counts once); `None` means no languages (count 0, not missing); names follow the 2021+ style so they match 2011 and the later years: `Objective-C`, `HTML/CSS`, `Bash/Shell`.

In [36]:
CHECKBOX_MAP = {"Java": "Java", "JavaScript": "JavaScript", "CSS": "HTML/CSS", "PHP": "PHP", "Python": "Python", "Objective-C": "Objective-C",
                "Ruby": "Ruby", "SQL": "SQL", "C#": "C#", "C++": "C++", "C": "C", "Perl": "Perl", "HTML5": "HTML/CSS", "None": None}
print(pd.DataFrame([[k, v] for k, v in CHECKBOX_MAP.items()], columns=["tick-box label (row 0)", "canonical name"]).to_string(index=False))
assert set(lang_labels[:-1]) == set(CHECKBOX_MAP), sorted(set(lang_labels[:-1]) ^ set(CHECKBOX_MAP))

tick-box label (row 0) canonical name
                  Java           Java
            JavaScript     JavaScript
                   CSS       HTML/CSS
                   PHP            PHP
                Python         Python
           Objective-C    Objective-C
                  Ruby           Ruby
                   SQL            SQL
                    C#             C#
                   C++            C++
                     C              C
                  Perl           Perl
                 HTML5       HTML/CSS
                  None            NaN


### Cell 5.3 - Alias dictionary for the write-ins
The 2011 dictionary, extended after reading the 2012 write-ins. Each rule is a pattern searched in the lowercased text; a match is *removed* before the next rule runs (so `vb.net` is not also `vb`); more specific rules come first. Every alias table row is saved to `language_alias_table_2012.csv`.

**Decisions (review me), same as 2011 unless noted:**
- **Languages are counted; tools are not.** Frameworks, platforms, databases and markup (jQuery, ASP.NET, Android, Flex, XML, MySQL, MongoDB, LaTeX...) are recognised but kept out of the list and the count.
- **Visual Basic stays split** (`VB.NET`, `Visual Basic 6`, `Visual Basic`); `VBA`, `VBScript` separate. `Object Pascal` -> `Delphi`; `PL/SQL`, `PL/pgSQL` separate from `SQL`.
- New in 2012 vs 2011: `plsql` also matches `PL/SQL`; `html4` / `css3` match `HTML/CSS`; `js` -> `JavaScript`; assembly dialects and chips (`asm`, `x86`, `ARM`, `68k`, `Z80`, `MIPS`...) -> `Assembly`; `xsl` -> `XSLT`; `Cold Fusion` -> `ColdFusion`; `google go` -> `Go`; `OpenEdge` -> `Progress ABL`; `vb9`/`vb8` -> `VB.NET`; `PowerScript` -> `PowerBuilder`; `VFP` -> `Visual FoxPro`.
- One-letter names (`R`, `D`, `C`) only count when they are the whole phrase.
- Commentary rows that need a human read are decided in `MANUAL` and shown in Cell 5.5 (the person says they are not proficient, or the text is a rant).
- A write-in of just `none`, `non` or `no` is treated as the `None` tick-box (count 0).
- Anything left over is kept as `unrecognized` for audit and never counted.

In [37]:
def _name_regex(name):
    return r"\b" + re.escape(name.lower()).replace(r"\ ", r"\s+") + r"\b"

LANG_RULES = [
    ("VB.NET", r"visual\s*basic\s*\.?\s*net|\bvb\s*\.?\s*net|\bvb\s*9\b|\bvb\s*8\b"),
    ("VBScript", r"\bvb\s*script|\bvbs\b"),
    ("VBA", r"\bvba\b"),
    ("Visual Basic 6", r"\bvb\s*6\b|visual\s*basic\s*6|classic\s*vb"),
    ("Visual Basic", r"visual\s*basic|\bvb\b"),
    ("PL/SQL", r"\bpl\s*/?\s*sql\b"),
    ("SQL", r"\bsql\b"),
    ("Objective-C", r"\bobj(?:ective|ectve)?[\s\-]*c\b"),
    ("C#", r"\bc\s?#"),
    ("C++", r"\bc\s?\+\+|\bcpp\b"),
    ("F#", r"\bf\s?#"),
    ("Java", r"\bjava\b"),
    ("JavaScript", r"\bjavascript\b|\becmascript\b"),
    ("PHP", r"\bphp\b"),
    ("Python", r"\bpython\b"),
    ("Ruby", r"\bruby\b"),
    ("Perl", r"\bperl\b"),
    ("HTML/CSS", r"\bx?html\s*\d*\b|\bcss\s*\d*\b"),
    ("PowerShell", r"powershell"),
    ("Bash/Shell", r"\bbash\b|\bshell?(?:\s+script\w*)?\b|\bksh\b|\bzsh\b|\bcsh\b|\bsh\b"),
    ("Assembly", r"(?:x86\s+|arm\s+|z/)?assembl(?:y|er)\b(?:\s+language)?"),
    ("Delphi", r"delphi|object\s*pascal|\boo-pascal\b"),
    ("Pascal", r"\bpascal\b"),
    ("ActionScript", r"action\s*scr?i?p?t\w*(?:\s*[123](?:\.0)?(?:\s*/\s*[123])*)?|actionsc\w*|\bas[123](?:-[123])?\b"),
    ("ColdFusion", r"coldfusion|\bcfml\b"),
    ("Lisp", r"\blisp\b"),
    ("MEL", r"\bmel\b"),
]
RARE_LANGS = ["Scheme", "Clojure", "Haskell", "Scala", "Erlang", "Lua", "Groovy", "OCaml", "Prolog", "Fortran", "COBOL", "Ada",
              "Tcl", "AWK", "Smalltalk", "Eiffel", "Forth", "Rexx", "MATLAB", "Mathematica", "Maple", "IDL", "SAS", "RPG", "ABAP",
              "PowerBuilder", "FoxPro", "LabVIEW", "AppleScript", "VHDL", "Verilog", "AutoHotkey", "AutoIt", "LSL", "Avisynth",
              "KRL", "Wasabi", "GeneXus", "LotusScript", "UniBasic", "UniQuery", "Nemerle", "Gosu", "Vala", "Brainfuck", "Befunge",
              "LOLCODE", "RealBasic", "CCScript", "DataFlex", "WebFocus", "Linc 4GL", "Forte 4GL", "Progress ABL",
              "JMP Scripting Language", "XSLT", "Lansa", "BASIC"]
LANG_RULES += [(n, _name_regex(n)) for n in RARE_LANGS]

TOOL_RULES = [
    ("ASP.NET", r"\basp\.net(?:-mvc)?"), ("ASP", r"\basp\b(?:\s*(?:classic|3\.0))?"), (".NET", r"(?<!\w)\.net\b"),
    ("jQuery", r"\bjquery\b"), ("JSP", r"\bjsp\b"), ("Symfony", r"\bsymfony\b"), ("Silverlight", r"\bsilverlight\b"),
    ("WPF", r"\bwpf\b"), ("WCF", r"\bwcf\b"), ("Entity Framework", r"entity-framework"), ("Grails", r"\bgrails\b"),
    ("Cocoa", r"\bcocoa\b"), ("Android", r"\bandroid\b"), ("Flex", r"(?:adobe\s+)?\bflex\b"), ("Flash", r"\bflash\b"),
    ("Excel", r"\bexcel\b"), ("NumPy", r"\bnumpy\b"), ("LaTeX", r"\blatex\b"), ("XML", r"\bxml\b"), ("XAML", r"\bxaml\b"),
    ("X3D", r"\bx3d\b"), ("XPath", r"\bxpath\b"), ("sed", r"\bsed\b"),
]
EXACT_LANG = {"r": "R", "d": "D", "j": "J", "go": "Go"}
EXACT_TOOL = {"make": "Make"}
# two answers where commentary contains language-like words; decided by hand and shown in Cell 5.5
MANUAL = {"css is not a programming language.": set(),
          "vb (why the fuck was this not a choice? c# snobs?)": {"Visual Basic"}}

COMPILED_LANG = [(n, re.compile(p)) for n, p in LANG_RULES]
COMPILED_TOOL = [(n, re.compile(p)) for n, p in TOOL_RULES]

def parse_writein(text):
    s = str(text).lower().strip().replace("’", "'")
    if s in MANUAL:
        return set(MANUAL[s]), set(), ["(manual override)"]
    langs, tools = set(), set()
    for name, rx in COMPILED_LANG:
        if rx.search(s):
            langs.add(name)
            s = rx.sub(" | ", s)
    for name, rx in COMPILED_TOOL:
        if rx.search(s):
            tools.add(name)
            s = rx.sub(" | ", s)
    leftovers = []
    for phrase in re.split(r"[|,;/&+()\[\]:!?]+", s):
        p = re.sub(r"[^a-z0-9#+.\- ]", "", phrase).strip(" .-")
        if not p:
            continue
        if p in EXACT_LANG:
            langs.add(EXACT_LANG[p])
        elif p in EXACT_TOOL:
            tools.add(EXACT_TOOL[p])
        else:
            leftovers.append(p)
    return langs, tools, leftovers

In [38]:
# ---- 2012 additions ----
LANG_RULES += [
    ("CoffeeScript", r"coff?ee?s+c?ript|coffescript"),
    ("JavaScript", r"\bjs\b"),
    ("XSLT", r"\bxsl\w*"),
    ("XQuery", r"\bxquery\b"),
    ("Racket", r"\bracket\b"),
    ("CUDA", r"\bcuda\b"),
    ("HLSL", r"\bhlsl\b"),
    ("QML", r"\bqml\b"),
    ("Octave", r"\boctave\b"),
    ("Scilab", r"\bscilab\b"),
    ("PL/pgSQL", r"\bpl\s*/?\s*pgsql\b"),
    ("PL/I", r"\bpl\s*/?\s*(?:1|i)\b"),
    ("ML", r"\bstandard\s+ml\b|\bmoscow\s+ml\b|\bsml\b"),
    ("Go", r"\bgoogle\s+go\b|\bgolang\b"),
    ("Haxe", r"\bhaxe\b"),
    ("Coq", r"\bcoq\b"),
    ("Isabelle", r"\bisabelle\b"),
    ("Apex", r"\bsalesforce\s+apex\b|\bapex\b"),
    ("MaxScript", r"\bmaxscript\b"),
    ("Max/MSP", r"\bmax\s*[/\-\s]\s*msp\b"),
    ("mIRC Script", r"\bmirc\b(?:\s+scripting(?:\s+language)?)?"),
    ("AutoIt", r"\bau3\b"),
    ("Progress ABL", r"\bprogress\s+(?:4gl|openedge|abl)\b"),
    ("Informix 4GL", r"\binformix\s+4gl\b"),
    ("PowerBuilder", r"\bpowerscript\b"),
    ("Visual FoxPro", r"\bvfp\b|visual\s+fox\s*pro"),
    ("Batch", r"\bbatch\b"),
    ("Erlang", r"\botp\b"),
    ("Phrogram", r"\bphrogram\b"), ("Pike", r"\bpike\b"), ("Squirrel", r"\bsquirrel\b"), ("Pawn", r"\bpawn\b"),
    ("SkookumScript", r"\bskookumscript\b"), ("Omnis Studio", r"\bomnis\s+studio\b"),
    ("DIBOL", r"\bdibol\b"), ("BLISS", r"\bbliss\b"), ("ASCET", r"\bascet\b"), ("HOL", r"\bhol\b"),
    ("SPSS", r"\bspss\b"), ("OpenCL", r"\bopencl\b"), ("GLSL", r"\bglsl\b"),
    ("Assembly", r"\basm\w*|\bnasm\b|\bx86(?:[\-\s]?64)?\b|\bx64\b|\b68k\b|\b68hc11\b|\barm\d*\b|\bmips\b|\bavr\d*\b|\bppc\b|\brisc\b"),
]
TOOL_RULES += [
    ("MySQL", r"\bmysql\b"), ("Redis", r"\bredis\b"), ("MongoDB", r"\bmongodb\b"), ("Elasticsearch", r"\belasticsearch\b"),
    ("Oracle", r"\boracle\b"), ("Node.js", r"\bnode\.?js\b|\bnode\b"), ("Django", r"\bdjango\b"), ("Magento", r"\bmagento\b"),
    ("Haml/Sass", r"\bhaml\b|\bsass\b"), ("MS Access", r"\bms\s+access\b"), ("Visual Studio", r"visual\s+studio(?:\s+\d+(?:\.\d+)?)?"),
    ("Struts", r"\bstruts\d?\b"), ("MVC", r"\bmvc\b"), ("Qt", r"\bqt\b"), ("Ajax", r"\bajax\b"), ("Google App Engine", r"google\s+app\s+engine"),
    ("Unix", r"\bunix\b"), ("XUL", r"\bxul\b"), ("OpenXML SDK", r"openxml-sdk"), ("SDK", r"\bsdk\b"),
    (".NET", r"\bdot\s*net\b"), ("ASP", r"classic[\s\-]*asp\b"), ("LaTeX", r"\btex\b"), ("Emacs", r"\bemacs\b"),
]
EXACT_LANG.update({"c": "C", "dm": "DM", "processing": "Processing"})
MANUAL.update({
    "learning php, c, c++, java, and a few others.  so i can't say i'm proficient in any.": set(),
    "learning c": set(),
    "those are not languages": set(),
})
COMPILED_LANG = [(n, re.compile(p)) for n, p in LANG_RULES]
COMPILED_TOOL = [(n, re.compile(p)) for n, p in TOOL_RULES]

# ---- second pass additions ----
LANG_RULES += [
    ("ColdFusion", r"cold\s+fusion"), ("APL", r"\bapl\b"), ("Rebol", r"\brebol\b"), ("Ragel", r"\bragel\b"),
    ("ChaiScript", r"\bchaiscript\b"), ("SQR", r"\bsqr\b|structured\s+query\s+report\w*"), ("ATL", r"\batl\b"),
    ("Xtend", r"\bxtend\b"), ("X++", r"\bx\+\+"), ("Fortran", r"\bfortan\b"), ("Self", r"\bself\b"),
    ("Progress ABL", r"\bopenedge\b"), ("Bash/Shell", r"\bshells\b"),
    ("Assembly", r"\bm68k\b|\bz80\b|\b6502\b|\b68000\b"),
]
TOOL_RULES += [("MXML", r"\bmxml\b"), ("CouchDB", r"\bcouchdb\b"), ("Puppet", r"\bpuppet\b"), ("Force.com", r"force\.com")]
MANUAL.update({"js, css, html...?  those are not \"languages\"!  the only people who refer to those as languages are people who arent smart enough to learn a real language.": set(),
               "learning c/c++, donno if i'm proficient yet.": set()})
COMPILED_LANG = [(n, re.compile(p)) for n, p in LANG_RULES]
COMPILED_TOOL = [(n, re.compile(p)) for n, p in TOOL_RULES]

In [39]:
NONE_PHRASES = {"none", "non", "no"}
alias_table = pd.DataFrame([[n, "language", p] for n, p in LANG_RULES] + [[n, "tool/markup (not counted)", p] for n, p in TOOL_RULES] +
                           [[v, "language (exact phrase only)", "^" + k + "$"] for k, v in EXACT_LANG.items()],
                           columns=["canonical_name", "kind", "pattern"])
print("alias rules:", len(alias_table), "| language names:", alias_table[alias_table.kind == "language"].canonical_name.nunique(),
      "| tool names:", alias_table[alias_table.kind.str.startswith("tool")].canonical_name.nunique())
for test in ["vb.net, vb6", "Objective-C (how dare you leave that out)! :-p", "r vb", "PIC, ARM7, AVR32, X86 and X64 assembly", "js, css, html...?", "HTML4 ;-)", "plsql"]:
    print(repr(test), "->", parse_writein(test))

alias rules: 199 | language names: 129 | tool names: 48
'vb.net, vb6' -> ({'Visual Basic 6', 'VB.NET'}, set(), [])
'Objective-C (how dare you leave that out)! :-p' -> ({'Objective-C'}, set(), ['how dare you leave that out', 'p'])
'r vb' -> ({'Visual Basic', 'R'}, set(), [])
'PIC, ARM7, AVR32, X86 and X64 assembly' -> ({'Assembly'}, set(), ['pic', 'and'])
'js, css, html...?' -> ({'JavaScript', 'HTML/CSS'}, set(), [])
'HTML4 ;-)' -> ({'HTML/CSS'}, set(), [])
'plsql' -> ({'PL/SQL'}, set(), [])


### Cell 5.4 - Parse every write-in and show the result

In [40]:
parsed = {idx: parse_writein(text) for idx, text in writein.items()}
lang_counts = {}
for idx, (langs, tools, left) in parsed.items():
    for name in langs:
        lang_counts[name] = lang_counts.get(name, 0) + 1
print(pd.Series(lang_counts).sort_values(ascending=False).head(40).to_frame("respondents").T.to_string())
print("distinct languages found in write-ins:", len(lang_counts))
write_none = {idx for idx, t in writein.items() if str(t).lower().strip(" .") in NONE_PHRASES}
print("write-ins that just say none:", len(write_none))
print("write-ins with no language recognised:", sum(1 for v in parsed.values() if not v[0]), "of", len(parsed))

             VB.NET  Delphi  ActionScript  Visual Basic  Groovy  Scala  Bash/Shell  ColdFusion  VBA  Haskell  Assembly   R  Clojure  Lua  PL/SQL  MATLAB  Fortran  Visual Basic 6  F#  Pascal  Lisp  XSLT  Tcl  PowerShell  HTML/CSS  VBScript  Scheme  COBOL  CoffeeScript  Smalltalk  VHDL  Prolog  LabVIEW  Ada  BASIC  Progress ABL  Erlang  RPG  Forth  Go
respondents      69      54            48            44      28     26          24          23   20       19        17  17       13   13      13      13       12              11  11      11    10     8    7           7         7         7       7      6             5          5     5       5        4    4      3             3       3    3      3   3
distinct languages found in write-ins: 88
write-ins that just say none: 2
write-ins with no language recognised: 63 of 589


### Cell 5.5 - Review the odd ones
(1) every write-in longer than 25 characters or with commentary punctuation, with its parse; (2) most common leftover phrases, in case a real language was missed. Any miss is fixed by adding a rule in Cell 5.3, never by editing data.

In [41]:
long_ones = [(idx, t) for idx, t in writein.items() if len(t) > 25 or "?" in t or "!" in t]
print("commentary-style write-ins:", len(long_ones))
for idx, t in long_ones:
    langs, tools, left = parsed[idx]
    print("row", idx, "|", repr(t[:90]))
    print("      languages:", sorted(langs), "| tools:", sorted(tools), "| leftover:", left[:4])
leftover_counts = {}
for langs, tools, left in parsed.values():
    for p in left:
        leftover_counts[p] = leftover_counts.get(p, 0) + 1
print()
print("most common leftover phrases:")
print(pd.Series(leftover_counts).sort_values(ascending=False).head(25).to_string())

commentary-style write-ins: 30
row 295 | 'Coffescript, haml, sass, that sort of thing'
      languages: ['CoffeeScript'] | tools: ['Haml/Sass'] | leftover: ['that sort of thing']
row 348 | 'fortran forth pl1 ada pascal delphi dbl dibol bliss z80 68000'
      languages: ['Ada', 'Assembly', 'BLISS', 'DIBOL', 'Delphi', 'Forth', 'Fortran', 'PL/I', 'Pascal'] | tools: [] | leftover: ['dbl']
row 552 | "VB and I'll kick your ass in FORTRAN."
      languages: ['Fortran', 'Visual Basic'] | tools: [] | leftover: ['and ill kick your ass in']
row 875 | 'Erlang, Object Pascal, Scheme, Common Lisp, HTML4 ;-)'
      languages: ['Delphi', 'Erlang', 'HTML/CSS', 'Lisp', 'Scheme'] | tools: [] | leftover: ['common']
row 1760 | 'Delphi, FORTRAN, Forth, BASIC, VB, Pascal'
      languages: ['BASIC', 'Delphi', 'Forth', 'Fortran', 'Pascal', 'Visual Basic'] | tools: [] | leftover: []
row 1915 | 'PIC, ARM7, AVR32, X86 and X64 assembly'
      languages: ['Assembly'] | tools: [] | leftover: ['pic', 'and']
row 2126 

### Cell 5.6 - Build `LanguageHaveWorkedWith` and the count
**Decisions:** the list is the distinct canonical names, sorted, joined with `;`. `LanguageHaveWorkedWith_count` is its length. It is `0` for people who ticked `None` (or wrote just "none") with no language; it is empty (unknown) for people who left the block blank **and** for people whose only answer was an unrecognisable write-in ("various", "lots"). Someone who ticked `None` and also named a language keeps the language. A name in both a tick-box and the write-in counts once.

Caution for the pooled model: 2012 offered 13 tick-boxes, 2021+ offer 40+, so the same count does not mean the same thing across years. Compare within a year or give the model `survey_year`.

In [42]:
tick_cols = lang_cols[:-1]
tick_labels = lang_labels[:-1]
final_lists, final_counts, none_flags = [], [], []
for idx in df.index:
    names = set()
    none_ticked = idx in write_none
    for col, lab in zip(tick_cols, tick_labels):
        if pd.notna(df.at[idx, col]):
            if lab == "None":
                none_ticked = True
            else:
                names.add(CHECKBOX_MAP[lab])
    if idx in parsed:
        names |= parsed[idx][0]
    final_lists.append(";".join(sorted(names, key=str.lower)) if names else np.nan)
    final_counts.append(len(names) if names else (0 if none_ticked else np.nan))
    none_flags.append(none_ticked)
language_list = pd.Series(final_lists, index=df.index, dtype=object)
language_count = pd.Series(final_counts, index=df.index, dtype=float)
none_ticked = pd.Series(none_flags, index=df.index)
block_answered = df[lang_cols].notna().any(axis=1)
print("rows:", len(df), "| count blank:", int(language_count.isna().sum()),
      "(block unanswered:", int((~block_answered).sum()), "| answered but only an unrecognisable write-in:", int((block_answered & language_list.isna() & ~none_ticked).sum()), ")")
print("'None' (tick or write-in):", int(none_ticked.sum()), "| of those with a language too:", int((none_ticked & language_list.notna()).sum()), "| count = 0 rows:", int((language_count == 0).sum()))
print("count distribution:")
print(language_count.value_counts().sort_index().to_string())
print("mean count: %.2f" % language_count.mean())
log("5.6 languages", "tick-boxes + parsed write-ins -> LanguageHaveWorkedWith list and count")

rows: 3740 | count blank: 17 (block unanswered: 15 | answered but only an unrecognisable write-in: 2 )
'None' (tick or write-in): 4 | of those with a language too: 0 | count = 0 rows: 4
count distribution:
0.0       4
1.0     279
2.0     465
3.0     646
4.0     852
5.0     611
6.0     364
7.0     245
8.0     134
9.0      69
10.0     34
11.0      9
12.0      8
14.0      1
15.0      1
16.0      1
mean count: 4.24
[LOG] 5.6 languages - tick-boxes + parsed write-ins -> LanguageHaveWorkedWith list and count


### Cell 5.7 - Sanity checks and audit files
No name spelled two ways; list round-trips to the count; audit file with every write-in as typed, what was found, what was ignored and what was left over.

In [43]:
all_names = sorted({n for s in language_list.dropna() for n in s.split(";")}, key=str.lower)
print("distinct canonical language names in the column:", len(all_names))
print(all_names)
lowered = {}
for n in all_names:
    lowered.setdefault(n.lower(), []).append(n)
assert all(len(v) == 1 for v in lowered.values()), "two spellings of the same name"
for idx in df.index:
    s = language_list[idx]
    n = 0 if pd.isna(s) else len(s.split(";"))
    c = language_count[idx]
    assert (pd.isna(c) and pd.isna(s)) or c == n, "count does not match list at row " + str(idx)
print("count matches list length for every row")
audit = pd.DataFrame({"source_row": list(parsed.keys()), "as_typed": [writein[i] for i in parsed],
                      "languages_found": [";".join(sorted(parsed[i][0])) for i in parsed],
                      "tools_not_counted": [";".join(sorted(parsed[i][1])) for i in parsed],
                      "unrecognized_text": [" | ".join(parsed[i][2]) for i in parsed]})
audit.to_csv("lang_writein_audit_2012.csv", index=False)
alias_table.to_csv("language_alias_table_2012.csv", index=False)
print("saved lang_writein_audit_2012.csv (%d rows) and language_alias_table_2012.csv (%d rules)" % (len(audit), len(alias_table)))

distinct canonical language names in the column: 95
['ABAP', 'ActionScript', 'Ada', 'Apex', 'APL', 'ASCET', 'Assembly', 'ATL', 'AutoIt', 'AWK', 'Bash/Shell', 'BASIC', 'Batch', 'Befunge', 'BLISS', 'Brainfuck', 'C', 'C#', 'C++', 'Clojure', 'COBOL', 'CoffeeScript', 'ColdFusion', 'CUDA', 'D', 'DataFlex', 'Delphi', 'DIBOL', 'Erlang', 'F#', 'Forth', 'Fortran', 'FoxPro', 'Go', 'Groovy', 'Haskell', 'HLSL', 'HTML/CSS', 'IDL', 'Informix 4GL', 'Java', 'JavaScript', 'LabVIEW', 'Lisp', 'Lua', 'Mathematica', 'MATLAB', 'Max/MSP', 'MaxScript', 'ML', 'Objective-C', 'OCaml', 'Octave', 'Omnis Studio', 'Pascal', 'Perl', 'PHP', 'PL/I', 'PL/pgSQL', 'PL/SQL', 'PowerBuilder', 'PowerShell', 'Processing', 'Progress ABL', 'Prolog', 'Python', 'QML', 'R', 'Racket', 'Rexx', 'RPG', 'Ruby', 'SAS', 'Scala', 'Scheme', 'Scilab', 'SkookumScript', 'Smalltalk', 'SQL', 'SQR', 'Tcl', 'UniBasic', 'UniQuery', 'VB.NET', 'VBA', 'VBScript', 'Verilog', 'VHDL', 'Visual Basic', 'Visual Basic 6', 'Visual FoxPro', 'X++', 'XQuery', 'XS

## Gate 6 - Single-answer questions (kept as they are)
### Cell 6.1 - One repair: Excel damaged the experience bands
The experience column contains `40944`, `41070` and `11` instead of band labels. Decoded as Excel date serials they are 5 Feb 2012 and 10 Jun 2012, i.e. `2-5` and `6-10` read as day-month; `11` is `11+` with the plus lost. (2011 had different serials, `41310` and `41435`, for the same two bands.) Only this column is repaired; every other single-answer column keeps its text exactly.

In [44]:
for serial in (40944, 41070):
    print(serial, "->", datetime.date(1899, 12, 30) + datetime.timedelta(days=serial))
EXP_FIX = {"<2": "<2", "40944": "2-5", "41070": "6-10", "11": "11+"}
print("before:", df[COL["exp"]].value_counts(dropna=False).to_dict())
unmapped = sorted(set(df[COL["exp"]].dropna()) - set(EXP_FIX))
assert not unmapped, unmapped
years_code = df[COL["exp"]].map(EXP_FIX)
exp_order = ["<2", "2-5", "6-10", "11+"]
print("after: ")
print(years_code.value_counts(dropna=False).reindex(exp_order + [np.nan]).to_string())
log("6.1 YearsCode", "Excel serials 40944->2-5, 41070->6-10 and bare 11->11+ repaired; no other single-answer column changed")

40944 -> 2012-02-05
41070 -> 2012-06-10
before: {'11': 1223, '41070': 1115, '40944': 1100, '<2': 302}
after: 
How many years of IT/Programming experience do you have?
<2       302.0
2-5     1100.0
6-10    1115.0
11+     1223.0
NaN        NaN
[LOG] 6.1 YearsCode - Excel serials 40944->2-5, 41070->6-10 and bare 11->11+ repaired; no other single-answer column changed


### Cell 6.2 - Value sets of the single-answer columns
Printed so you can see exactly what stays. No merging, no imputation, blanks stay empty. Country keeps regions such as `Other Europe` as they are.

In [45]:
for k in ["country", "state", "age", "industry", "size", "occ", "project", "os", "sat", "careers_aware", "careers_profile", "careers_why", "spend", "rep", "site"]:
    s = df[COL[k]]
    print("---", k, "|", COL[k][:70], "| distinct:", s.nunique(), "| blank:", int(s.isna().sum()))
    if s.nunique() <= 15:
        print(s.value_counts(dropna=False).to_string())

--- country | What Country or Region do you live in? | distinct: 20 | blank: 0
--- state | Which US State or Territory do you live in? | distinct: 52 | blank: 2490
--- age | How old are you? | distinct: 8 | blank: 0
How old are you?
25-29    1260
30-34     827
20-24     789
35-39     403
40-50     315
< 20       69
51-60      58
>60        19
--- industry | How would you best describe the industry you currently work in? | distinct: 12 | blank: 0
How would you best describe the industry you currently work in?
Software Products          1205
Web Services                763
Consulting                  450
Other                       390
Finance / Banking           251
Education                   141
Manufacturing               127
Healthcare                  122
Retail                       87
Gaming                       78
Advertising                  76
Foundation / Non-Profit      50
--- size | Which best describes the size of your company? | distinct: 6 | blank: 0
Which best describe

## Gate 7 - Headers and assembly
### Cell 7.1 - Header map
**Decisions (review me):** names confirmed in your shared list are `Country`, `Age`, `YearsCode`, `OrgSize`, `DevType`, `LanguageHaveWorkedWith`, `OpSys`, `JobSat`, `CompTotal`, `ConvertedCompYearly`; the others use the same proposed names as the 2011 notebook so the two years line up. I could not see your Canonical Mapping spreadsheet, so check the proposed names against it in this one table. 2012-only questions (Stack Overflow Careers, advertising, reputation, extra site) get proposed names and stay as typed. `ResponseId` is the row number in the 2012 file (2012 has no respondent id). There is no `RecommendationActedUpon` column in 2012.

In [46]:
AD_NAMES = ["AdsRelevant", "AdsEntertaining", "AdsInformative", "AdsClickInteresting", "AdsTookTrialOrBought", "AdsUseAdBlocker"]
a, b = BLOCKS["ads"]
assert len(AD_NAMES) == b - a
HEADER_MAP = [
    ("ResponseId", "proposed", "(none in 2012)", "row number of the respondent in the 2012 file"),
    ("survey_year", "proposed", "(added)", "constant 2012"),
    ("Country", "confirmed", COL["country"], "2012 values are regions as well as countries (e.g. 'Other Europe'), kept as is"),
    ("USState", "proposed", COL["state"], "only asked of US respondents"),
    ("Age", "confirmed", COL["age"], "text bands, kept as typed"),
    ("YearsCode", "confirmed", COL["exp"], "text bands '<2','2-5','6-10','11+' after Excel repair; question says IT/programming experience"),
    ("Industry", "proposed", COL["industry"], ""),
    ("OrgSize", "confirmed", COL["size"], "text bands incl. 'Student' and 'Other (not working...)'"),
    ("DevType", "confirmed", COL["occ"], "single answer in 2012; later years may be ';' lists"),
    ("PurchaseInvolvement", "proposed", RAW[7] + " (cols 7-13)", "';' list of options ticked"),
    ("PurchaseTypes", "proposed", RAW[14] + " (cols 14-19)", "';' list"),
    ("BudgetOutsideExpenditure", "proposed", COL["budget"], "single text bracket; 'Don't know' kept as a value; no conflicting rows in 2012"),
    ("ProjectType", "proposed", COL["project"], ""),
    ("LanguageHaveWorkedWith", "confirmed", RAW[22] + " (cols 22-36)", "';' list of canonical names; tick-boxes + parsed write-ins; 2012 wording is 'proficient in'"),
    ("LanguageHaveWorkedWith_count", "proposed", "(derived)", "number of distinct languages; 0 if 'None'; blank if block unanswered or unrecognisable"),
    ("OpSys", "confirmed", COL["os"], "text as typed"),
    ("JobSat", "confirmed", COL["sat"], "text answers in 2012; later years use a 0-10 score, harmonise later"),
    ("CompTotal", "confirmed", COL["salary"], "salary BAND text, kept as typed"),
    ("ConvertedCompYearly", "confirmed", "(derived from " + COL["salary"][:40] + "...)", "annual USD band midpoint; includes bonus; top band set by CONFIG"),
    ("CareersAware", "proposed", COL["careers_aware"], "2012-only question, kept as typed"),
    ("CareersProfile", "proposed", COL["careers_profile"], "2012-only question, kept as typed"),
    ("CareersNoProfileReason", "proposed", COL["careers_why"], "2012-only; 'Other (please specify)' kept as that text"),
    ("CareersNoProfileReasonOther", "proposed", "(free text next to " + str(IDX["careers_why"]) + ")", "2012-only free text, kept as typed"),
    ("TechProductsOwned", "proposed", RAW[44] + " (cols 44-63)", "';' list; free-text 'Other' becomes the label Other"),
] + [(n, "proposed", RAW[a] + " / " + str(option_labels.iloc[a + i]) if i == 0 else str(option_labels.iloc[a + i]), "agree-disagree scale, text as typed")
     for i, n in enumerate(AD_NAMES)] + [
    ("PersonalTechSpend12m", "proposed", COL["spend"], "text bands, kept as typed"),
    ("AdvertisersRecalled", "proposed", COL["ad_recall"], "free text, kept as typed"),
    ("SOReputation", "proposed", COL["rep"], "text as typed (includes 'Don't have an account')"),
    ("SitesFrequentedMost", "proposed", COL["site"], "text as typed; includes a real answer 'None' - load the CSV with keep_default_na=False, na_values=['']"),
    ("SitesFrequentedMostOther", "proposed", "(free text next to " + str(IDX["site"]) + ")", "free text, kept as typed"),
]
header_map = pd.DataFrame(HEADER_MAP, columns=["canonical_name", "status", "original_header_or_span", "note"])
print(header_map.status.value_counts().to_dict())
print(header_map[["canonical_name", "status", "note"]].to_string(index=False, max_colwidth=90))

{'proposed': 25, 'confirmed': 10}
              canonical_name    status                                                                                       note
                  ResponseId  proposed                                              row number of the respondent in the 2012 file
                 survey_year  proposed                                                                              constant 2012
                     Country confirmed             2012 values are regions as well as countries (e.g. 'Other Europe'), kept as is
                     USState  proposed                                                               only asked of US respondents
                         Age confirmed                                                                  text bands, kept as typed
                   YearsCode confirmed text bands '<2','2-5','6-10','11+' after Excel repair; question says IT/programming exp...
                    Industry  proposed                  

### Cell 7.2 - Assemble the canonical table
Columns in questionnaire order with the shared names. Blanks stay blank; nothing is imputed, merged or encoded.

In [47]:
out = pd.DataFrame(index=df.index)
out["ResponseId"] = df.index.astype(int)
out["survey_year"] = 2012
out["Country"] = df[COL["country"]]
out["USState"] = df[COL["state"]]
out["Age"] = df[COL["age"]]
out["YearsCode"] = years_code
out["Industry"] = df[COL["industry"]]
out["OrgSize"] = df[COL["size"]]
out["DevType"] = df[COL["occ"]]
out["PurchaseInvolvement"] = purchase_involvement
out["PurchaseTypes"] = purchase_types
out["BudgetOutsideExpenditure"] = budget
out["ProjectType"] = df[COL["project"]]
out["LanguageHaveWorkedWith"] = language_list
out["LanguageHaveWorkedWith_count"] = language_count
out["OpSys"] = df[COL["os"]]
out["JobSat"] = df[COL["sat"]]
out["CompTotal"] = df[COL["salary"]]
label_to_mid = {b[0]: b[3] for b in BANDS}
out["ConvertedCompYearly"] = df[COL["salary"]].map(label_to_mid).astype(float)
out["CareersAware"] = df[COL["careers_aware"]]
out["CareersProfile"] = df[COL["careers_profile"]]
out["CareersNoProfileReason"] = df[COL["careers_why"]]
out["CareersNoProfileReasonOther"] = df[COL["careers_why_other"]]
out["TechProductsOwned"] = tech_products
for i, n in enumerate(AD_NAMES):
    out[n] = df[RAW[BLOCKS["ads"][0] + i]]
out["PersonalTechSpend12m"] = df[COL["spend"]]
out["AdvertisersRecalled"] = df[COL["ad_recall"]]
out["SOReputation"] = df[COL["rep"]]
out["SitesFrequentedMost"] = df[COL["site"]]
out["SitesFrequentedMostOther"] = df[COL["site_other"]]
assert list(out.columns) == list(header_map.canonical_name), "header map and table columns differ"
print("shape:", out.shape)
print(out.head(3).T.to_string(max_colwidth=60))

shape: (3740, 35)
                                                                                        1                                                            2                                                            3
ResponseId                                                                              1                                                            2                                                            3
survey_year                                                                          2012                                                         2012                                                         2012
Country                                                                             India                                                      Germany                                               United Kingdom
USState                                                                               NaN                                             

### Cell 7.3 - Validate
Right row count, no encoded columns, no empty strings, list columns well-formed, single-answer text unchanged from the file, every row has a salary, missing values left as missing.

In [48]:
assert len(out) == len(df)
assert out["ConvertedCompYearly"].notna().all(), "a kept row has no salary"
list_cols = ["PurchaseInvolvement", "PurchaseTypes", "LanguageHaveWorkedWith", "TechProductsOwned"]
for c in list_cols:
    s = out[c].dropna()
    assert not s.str.startswith(";").any() and not s.str.endswith(";").any() and not s.str.contains(";;").any(), c
    assert all(len(v.split(";")) == len(set(v.split(";"))) for v in s), "duplicate item in " + c
    print(c, "| blank:", int(out[c].isna().sum()), "| mean items: %.2f" % s.str.split(";").str.len().mean())
text_cols = [c for c in out.columns if not pd.api.types.is_numeric_dtype(out[c])]
for c in text_cols:
    assert not (out[c].dropna().str.strip() == "").any(), "empty string in " + c
numeric_cols = [c for c in out.columns if c not in text_cols]
na_like = {"None", "N/A", "NA", "n/a", "null", "NULL", "NaN", "nan"}
hits = {c: int(out[c].isin(na_like).sum()) for c in text_cols}
print("cells holding NA-like TEXT that is a real answer:", {k: v for k, v in hits.items() if v})
print("numeric columns:", numeric_cols)
for k, c in [("country", "Country"), ("age", "Age"), ("size", "OrgSize"), ("occ", "DevType"), ("os", "OpSys"), ("sat", "JobSat"), ("salary", "CompTotal")]:
    assert out[c].equals(df[COL[k]]), c + " differs from the file"
print("single-answer columns equal the file exactly (Country, Age, OrgSize, DevType, OpSys, JobSat, CompTotal)")
print("missing per column:")
print(out.isna().sum()[out.isna().sum() > 0].to_string())
print("any one-hot style 0/1 columns:", [c for c in numeric_cols if set(out[c].dropna().unique()) <= {0, 1} and c != "survey_year"])

PurchaseInvolvement | blank: 0 | mean items: 1.77
PurchaseTypes | blank: 1022 | mean items: 2.54
LanguageHaveWorkedWith | blank: 21 | mean items: 4.24
TechProductsOwned | blank: 160 | mean items: 3.17
cells holding NA-like TEXT that is a real answer: {'CareersNoProfileReasonOther': 1, 'AdvertisersRecalled': 67, 'SitesFrequentedMost': 31}
numeric columns: ['ResponseId', 'survey_year', 'LanguageHaveWorkedWith_count', 'ConvertedCompYearly']
single-answer columns equal the file exactly (Country, Age, OrgSize, DevType, OpSys, JobSat, CompTotal)
missing per column:
USState                         2490
PurchaseTypes                   1022
BudgetOutsideExpenditure        1022
LanguageHaveWorkedWith            21
LanguageHaveWorkedWith_count      17
OpSys                              6
CareersNoProfileReason          2493
CareersNoProfileReasonOther     3566
TechProductsOwned                160
AdsRelevant                      157
AdsEntertaining                  158
AdsInformative             

### Cell 7.4 - Save and round-trip check

In [49]:
out.to_csv("canonical_2012.csv", index=False)
header_map.to_csv("header_map_2012.csv", index=False)
back = pd.read_csv("canonical_2012.csv", keep_default_na=False, na_values=[""])
assert back.shape == out.shape and list(back.columns) == list(out.columns)
assert (back.isna().values == out.isna().values).all(), "missing-value pattern changed on save"
for c in out.columns:
    if not pd.api.types.is_numeric_dtype(out[c]):
        assert (back[c].fillna("").values == out[c].fillna("").values).all(), "text changed on save in " + c
    else:
        assert np.allclose(back[c].fillna(-1).values, out[c].fillna(-1).values), "numbers changed on save in " + c
print("saved canonical_2012.csv | on disk:", back.shape, "| in memory:", out.shape, "| round trip identical")
log("7.4 save", "canonical_2012.csv: " + str(out.shape[0]) + " rows x " + str(out.shape[1]) + " columns, not encoded")
print()
print("row log:")
for step, b0, a0 in row_log:
    print("  %-48s %5d -> %5d" % (step, b0, a0))
print()
print("decision log:")
for step, text in decision_log:
    print(" ", step, "|", text)

saved canonical_2012.csv | on disk: (3740, 35) | in memory: (3740, 35) | round trip identical
[LOG] 7.4 save - canonical_2012.csv: 3740 rows x 35 columns, not encoded

row log:
  1.6 option-label row moved out of data            6244 ->  6243
  2.1 junk rows (only Country/State answered)       6243 ->  5964
  3.2 salary band blank                             5964 ->  5237
  3.2 'Rather not say' band                         5237 ->  4255
  3.2 Student / Unemployed band                     4255 ->  3740

decision log:
  1.1 encoding | file read with encoding=cp1252 (8 non-ASCII byte(s) in the file)
  1.2 load | loaded 6244 rows x 75 columns, all as text, default NA parsing off
  1.4 strip | 2 column name(s) had surrounding whitespace and were stripped
  1.5 blanks | 283328 empty/whitespace cells converted to NaN; 160 literal 'None'-style cells kept as text
  1.6 option row | row 0 moved to option_labels (rows 6244 -> 6243)
  2.1 junk rows | 279 rows removed
  3.2 salary rows | rows witho